# 🎭 基于对抗生成网络的人脸生成系统

> 课程设计项目 | StyleGAN2 + InterFaceGAN 潜空间操控

### ✨ 功能
- 🔀 **随机生成** — 1024×1024 高保真人脸, 每张都是独一无二的
- 👫 **性别控制** — 拖滑块实时切换男女, 保持身份不变
- 🎞️ **渐变 Morphing** — 从男性到女性的连续过渡动画
- 📐 **截断分析** — 控制生成质量 vs 多样性的权衡

### 🔬 核心技术
| 组件 | 说明 |
|------|------|
| 生成模型 | NVIDIA StyleGAN2-ADA (官方), FFHQ数据集预训练 |
| 性别控制 | InterFaceGAN 方法: W空间SVM + 线性方向操控 |
| 分类器 | OpenCV 预训练性别分类器 (~95%准确率) |
| 框架 | PyTorch + CUDA, 纯推理无需训练 |

## 1️⃣ 环境初始化

In [ ]:
# ⚡ 必须在所有 import 之前: 跳过 CUDA 编译, 直接用纯 PyTorch ops
from src import silence_ops

import os, torch, dnnlib, legacy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
import warnings; warnings.filterwarnings('ignore')
%matplotlib inline
device = 'cuda'
print(f'🚀 {torch.cuda.get_device_name(0)} | PyTorch {torch.__version__}\n')

In [ ]:
# 加载预训练模型
model_path = 'weights/ffhq.pkl'
if not os.path.exists(model_path):
    print('📥 首次运行: 下载权重 (~300MB)...')
    import requests, tqdm
    r = requests.get('https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl', stream=True)
    os.makedirs('weights', exist_ok=True)
    with open(model_path, 'wb') as f:
        for chunk in tqdm.tqdm(r.iter_content(8192), total=int(r.headers.get('content-length',0))//8192):
            f.write(chunk)

with dnnlib.util.open_url(model_path) as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device).eval()

print(f'✅ StyleGAN2-FFHQ | {G.img_resolution}×{G.img_resolution} | z_dim={G.z_dim} | W layers={G.mapping.num_ws}')

In [5]:
# 加载性别方向 (如未计算, 用演示方向)
if os.path.exists('weights/gender_dir_official.pt'):
    gender_dir = torch.load('weights/gender_dir_official.pt', map_location=device)
    print(f'✅ 已加载性别方向: shape={gender_dir.shape}')
else:
    torch.manual_seed(42)
    gender_dir = torch.randn(1, G.mapping.num_ws, G.z_dim, device=device)
    gender_dir = gender_dir / gender_dir.norm(p=2, dim=-1, keepdim=True)
    print('⚠️ 演示方向 (运行 compute_gender.py 获取真实方向)')

NameError: name 'os' is not defined

## 2️⃣ 核心函数

In [ ]:
@torch.no_grad()
def gen(z=None, seed=None, psi=0.7):
    """z → W → Image"""
    if seed is not None: torch.manual_seed(seed)
    if z is None: z = torch.randn(1, G.z_dim, device=device)
    ws = G.mapping(z, None, truncation_psi=psi)
    return G.synthesis(ws, noise_mode='const'), ws, z

def img2arr(t):
    """tensor [-1,1] → numpy [0,255] uint8"""
    return ((t.squeeze(0).permute(1,2,0).cpu().detach()+1)*127.5).clamp(0,255).to(torch.uint8).numpy()

def show(t, title='', s=6):
    plt.figure(figsize=(s,s)); plt.imshow(img2arr(t)); plt.axis('off')
    if title: plt.title(title, fontsize=13)
    plt.tight_layout(); plt.show()

def grid(imgs, titles=None, cols=4, fs=(16,12)):
    rows = (len(imgs)+cols-1)//cols
    fig, ax = plt.subplots(rows, cols, figsize=fs)
    ax = ax.flatten() if rows*cols>1 else [ax]
    for i, a in enumerate(ax):
        if i < len(imgs):
            a.imshow(img2arr(imgs[i]))
            if titles and i < len(titles):
                a.set_title(titles[i], fontsize=9)
        a.axis('off')
    plt.tight_layout(); plt.show()

print('✅ 函数就绪')

## 3️⃣ 随机人脸生成

In [ ]:
print('🎲 6 张 1024×1024 随机人脸\n')
imgs = [gen(seed=i)[0] for i in range(6)]
grid(imgs, [f'#{i+1}' for i in range(6)], cols=3, fs=(14,10))

## 4️⃣ 交互式性别控制

> **原理**: W空间中性别方向 $\mathbf{d}$ → $W' = W + \alpha \cdot \mathbf{d}$

In [ ]:
# 锁定一个身份
torch.manual_seed(42)
base_ws = G.mapping(torch.randn(1, G.z_dim, device=device), None)

slider = widgets.FloatSlider(value=0, min=-5, max=5, step=0.2, description='性别:',
    style={'description_width':'50px'}, layout=widgets.Layout(width='460px'))
live_label = widgets.Label(value='当前: 😐 中性')
out = widgets.Output()

def update(change):
    a = change['new']
    if a > 0.5:  live_label.value = f'当前: 👩 女性 (α={a:+.1f})'
    elif a < -0.5: live_label.value = f'当前: 👨 男性 (α={a:+.1f})'
    else: live_label.value = f'当前: 😐 中性 (α={a:+.1f})'
    img = G.synthesis(base_ws + a * gender_dir, noise_mode='const')
    with out: clear_output(wait=True); show(img, live_label.value, s=7)

slider.observe(update, 'value')
update({'new':0})
display(widgets.VBox([slider, live_label, out]))

### 性别渐变 — 从男到女连续 Morphing

In [ ]:
alphas = np.linspace(-5, 5, 11)
walk = [G.synthesis(base_ws + a*gender_dir, noise_mode='const') for a in alphas]
titles = [f'{"👨" if a<-1 else "👩" if a>1 else "😐"} α={a:+.1f}' for a in alphas]
grid(walk, titles, cols=6, fs=(22,8))
print('👈 男性 ←————————————→ 女性 👉')

## 5️⃣ 多身份性别对比

> 6 个不同身份 × 3 个性别级别 = 验证身份一致的性别控制

In [ ]:
all_imgs, all_titles = [], []
for pid in range(6):
    torch.manual_seed(pid*13+7)
    ws = G.mapping(torch.randn(1, G.z_dim, device=device), None)
    for lbl, a in [('👨男', -4), ('😐中', 0), ('👩女', +4)]:
        all_imgs.append(G.synthesis(ws+a*gender_dir, noise_mode='const'))
        all_titles.append(f'ID#{pid+1} {lbl}')
grid(all_imgs, all_titles, cols=3, fs=(12,18))

## 6️⃣ 截断系数 ψ (Truncation Trick)

> ψ 越小 → 越接近"平均脸"(质量↑多样性↓); ψ=1 → 完全随机

In [ ]:
torch.manual_seed(999)
z = torch.randn(1, G.z_dim, device=device)
trunc = [gen(z=z, psi=p)[0] for p in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]]
grid(trunc, [f'ψ={p}' for p in [0.0,0.2,0.4,0.6,0.8,1.0]], cols=3, fs=(15,10))

## 7️⃣ 批量随机生成 (可导出)

> 快速生成一批人脸, 可保存到本地

In [ ]:
os.makedirs('outputs', exist_ok=True)
print('📸 生成 12 张并保存...')
for i in range(12):
    img, _, _ = gen(seed=i*100)
    Image.fromarray(img2arr(img)).save(f'outputs/face_{i:03d}.png')
    
# 显示 4x3 网格
saved = [gen(seed=i*100)[0] for i in range(12)]
grid(saved, [f'#{i+1}' for i in range(12)], cols=4, fs=(18,14))
print('💾 已保存到 outputs/ 目录')

## 8️⃣ 技术原理 (答辩参考)

### GAN 训练目标

$$\min_G \max_D \; \mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))]$$

### StyleGAN2 核心创新

1. **Mapping Network** z→W: 将噪声映射到解缠潜空间
2. **Weight Demodulation**: 消除"水滴"伪影 (StyleGAN2相比StyleGAN1的主要改进)
3. **Noise Injection**: 每层加随机噪声 → 微观细节(毛孔、雀斑)
4. **Path Length Regularization**: W空间更平滑 → 插值更自然

### InterFaceGAN 潜空间操控

$$\mathbf{w}' = \mathbf{w} + \alpha \cdot \mathbf{n}_{attribute}$$

- 在W空间训练线性SVM → 找到语义属性的决策超平面
- 超平面法向量 $\mathbf{n}$ 即操控方向
- 适用于: 性别、年龄、表情、姿态、发色等

## 9️⃣ 总结

| 指标 | 数值 |
|------|------|
| 输出分辨率 | 1024×1024 |
| 预训练数据集 | FFHQ (70K 高清人脸) |
| 模型参数量 | ~30M |
| 生成速度 | ~0.5s/张 (RTX 4060) |
| 性别分类器 | OpenCV CNN (轻量, <2MB) |
| 方向计算 | Linear SVM on W-space |

**提交**: `jupyter nbconvert --to html --execute demo.ipynb`